In [1]:
import numpy as np
from ase.io import read
from mace.calculators import mace_mp 
from sklearn.decomposition import PCA
import chemiscope
from IPython.display import display

# 1. Load the structures
structures = read("dataset.xyz", index=":")

# 2. Setup MACE to extract embeddings
print("Loading MACE model...")
# Using the foundational MP model. Swap to MACE(model="your_model.model") if custom.
calc = mace_mp(model="small", device="cpu", default_dtype="float32")

print("Computing MACE embeddings (this takes a moment)...")
mace_features = []
for s in structures:
    # Extract the (n_atoms x embedding_dimension) node tensor
    node_embeddings = calc.get_descriptors(s)
    
    # Pool the atomic embeddings into a single global representation for the structure
    global_embedding = np.mean(node_embeddings, axis=0)
    mace_features.append(global_embedding)

# Convert to a 2D numpy array: (n_structures, n_features)
mace_features = np.array(mace_features)

# 3. Dimensionality Reduction (PCA)
print("Running PCA to reduce MACE embeddings to a 2D phase space...")
pca = PCA(n_components=2) # Extract the top two principal components
pca_embeddings = pca.fit_transform(mace_features)

# Output the explained variance to see how much information is captured by these 2 axes
explained_var = pca.explained_variance_ratio_
print(f"Explained Variance: PC1 = {explained_var[0]:.1%}, PC2 = {explained_var[1]:.1%}")

# Extract PCA coordinates into lists for Chemiscope
pca1 = pca_embeddings[:, 0].tolist()
pca2 = pca_embeddings[:, 1].tolist()

# 4. Extract Physical Properties
# Assuming energy and forces are already stored in the .xyz file
properties = {
    "Structure_ID": list(range(len(structures))),
    "Energy": [s.get_potential_energy() for s in structures],
    "Max_Force": [abs(s.get_forces()).max() for s in structures],
    
    # Map the PCA coordinates
    "PCA_1": pca1,
    "PCA_2": pca2
}

# 5. Visualize in Chemiscope
print("Launching Chemiscope...")
widget = chemiscope.show(
    structures=structures,
    properties=properties,
    settings=chemiscope.quick_settings(
        x="PCA_1",             
        y="PCA_2",             
        map_color="Energy"  
    )
)

display(widget)

/home/md/micromamba/envs/md_env/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Loading MACE model...
Using Materials Project MACE for MACECalculator with /home/md/.cache/mace/20231210mace128L0_energy_epoch249model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/md/micromamba/envs/md_env/lib/python3.11/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Computing MACE embeddings (this takes a moment)...
Running PCA to reduce MACE embeddings to a 2D phase space...
Explained Variance: PC1 = 52.1%, PC2 = 23.0%
Launching Chemiscope...


<ChemiscopeWidget(meta={'name': ' '}, settings={'map': {'joinPoints': False, 'x': {'property': 'PCA_1'}, 'y': …